<a href="https://colab.research.google.com/github/AmandhaPanagoda/AmandhaPanagoda/blob/main/notebooks/03_encoder_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
# install rdkit for Morgan fingerprint generation
!pip install rdkit --quiet
!pip install requests --quiet

In [59]:
import os
import json
import time
import joblib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # suppress rdkit warnings

sns.set_theme(style='whitegrid')
torch.manual_seed(42)
np.random.seed(42)

DATA_DIR   = '/content/drive/MyDrive/FYP Research/Notebooks/Data'
PROC_DIR   = '/content/drive/MyDrive/FYP Research/Notebooks/Processed'
PREP_DIR   = os.path.join(PROC_DIR, 'preprocessed')
MODEL_DIR  = os.path.join(PROC_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

In [60]:
# helper functions
def find_file(patterns, data_dir=DATA_DIR):
  if not os.path.isdir(data_dir):
    return None
  for pat in patterns:
    for f in os.listdir(data_dir):
      if pat.lower() in f.lower():
        return os.path.join(data_dir, f)
  return None

In [61]:
# load omic matrices
expr_scaled = np.load(os.path.join(PREP_DIR, 'expr_scaled.npy'))
cnv_scaled  = np.load(os.path.join(PREP_DIR, 'cnv_scaled.npy'))
mut_scaled  = np.load(os.path.join(PREP_DIR, 'mut_scaled.npy'))

print('loaded omic matrices:')
print(f'  expression: {expr_scaled.shape}  dtype={expr_scaled.dtype}')
print(f'  CNV:        {cnv_scaled.shape}  dtype={cnv_scaled.dtype}')
print(f'  mutation:   {mut_scaled.shape}  dtype={mut_scaled.dtype}')

N_CELL_LINES = expr_scaled.shape[0]
print(f'\ncell lines: {N_CELL_LINES}')

loaded omic matrices:
  expression: (713, 2000)  dtype=float32
  CNV:        (713, 2000)  dtype=float32
  mutation:   (713, 292)  dtype=float32

cell lines: 713


In [62]:
# load cohort ids - defines the row order for all matrices
with open(os.path.join(PREP_DIR, 'cohort_ids.json')) as f:
  cohort_ids = json.load(f)

# build a lookup: ModelID -> row index
cellline_to_idx = {mid: i for i, mid in enumerate(cohort_ids)}
print(f'cohort_ids loaded: {len(cohort_ids)} entries')
print(f'first 5: {cohort_ids[:5]}')

cohort_ids loaded: 713 entries
first 5: ['ACH-000001', 'ACH-000002', 'ACH-000004', 'ACH-000006', 'ACH-000007']


In [63]:
# load master_pairs - the (cell line, drug) training labels
master_pairs = pd.read_parquet(os.path.join(PROC_DIR, 'master_pairs.parquet'))

print(f'master_pairs: {master_pairs.shape}')
print(f'  cell lines: {master_pairs["ModelID"].nunique()}')
print(f'  drugs:      {master_pairs["DRUG_ID"].nunique()}')
print(f'  LN_IC50 range: [{master_pairs["LN_IC50"].min():.3f}, {master_pairs["LN_IC50"].max():.3f}]')
print(f'  LN_IC50 NaN: {master_pairs["LN_IC50"].isnull().sum()}')
print()
display(master_pairs.head(3))

master_pairs: (180514, 9)
  cell lines: 713
  drugs:      295
  LN_IC50 range: [-8.748, 13.820]
  LN_IC50 NaN: 0



,ModelID,DRUG_ID,DRUG_NAME,LN_IC50,AUC,TCGA_DESC,SANGER_MODEL_ID,COSMIC_ID,CELL_LINE_NAME
0,ACH-001711,1003,Camptothecin,-1.463887,0.930220,MB,SIDM01132,683667,PFSK-1
1,ACH-000052,1003,Camptothecin,-4.869455,0.614970,UNCLASSIFIED,SIDM00848,684052,A673
2,ACH-000087,1003,Camptothecin,-5.142961,0.582439,UNCLASSIFIED,SIDM01111,684072,SK-ES-1


In [64]:
# load gene lists for reference
with open(os.path.join(PREP_DIR, 'gene_lists.json')) as f:
    gene_lists = json.load(f)

print('gene lists:')
print(f'  expression: {len(gene_lists["expression"])} genes')
print(f'  CNV:        {len(gene_lists["cnv"])} genes')
print(f'  mutation:   {len(gene_lists["mutation"])} genes (+ 1 burden = {mut_scaled.shape[1]} total columns)')

# input dimensions for encoder construction
EXPR_DIM = expr_scaled.shape[1]
CNV_DIM  = cnv_scaled.shape[1]
MUT_DIM  = mut_scaled.shape[1]
print(f'\nencoder input dimensions: expr={EXPR_DIM}, cnv={CNV_DIM}, mut={MUT_DIM}')

gene lists:
  expression: 2000 genes
  CNV:        2000 genes
  mutation:   291 genes (+ 1 burden = 292 total columns)

encoder input dimensions: expr=2000, cnv=2000, mut=292


The mutation gene list JSON tracks 291 genes by name - those are the COSMIC Tier 1 gene.
But the actual saved numpy array mut_scaled has 292 columns because in preprocessing stage we have appended the mutation burden scalar as the 292nd column.

*The burden column has no gene name (it's a computed summary statistic, not a specific gene), so it's not in gene_lists.json.*

## Drug SMILES and Morgan fingerprints

In [65]:
# inspect the drug list from GDSC2
master_pairs['DRUG_NAME'] = master_pairs['DRUG_NAME'].str.strip() # strip leading and trailing whitespaces from drug names
drugs = master_pairs[['DRUG_ID', 'DRUG_NAME']].drop_duplicates().reset_index(drop=True)
print(f'unique drugs in training set: {len(drugs)}')
print()
print('shape of drugs:')
display(drugs.head(10))

unique drugs in training set: 295

shape of drugs:


,DRUG_ID,DRUG_NAME
0,1003,Camptothecin
1,1004,Vinblastine
2,1005,Cisplatin
3,1006,Cytarabine
4,1007,Docetaxel
5,1008,Methotrexate
6,1009,Tretinoin
7,1010,Gefitinib
8,1011,Navitoclax
9,1012,Vorinostat


In [66]:
# drugs with missing names
missing_name = drugs['DRUG_NAME'].isnull().sum()
print(f'drugs with missing DRUG_NAME: {missing_name}')
if missing_name > 0:
  print(drugs[drugs['DRUG_NAME'].isnull()])

drugs with missing DRUG_NAME: 0


In [67]:
!pip install pubchempy --quiet

In [68]:
from pubchempy import get_compounds

SMILES_CACHE = os.path.join(PROC_DIR, "drug_smiles.json")

def fetch_smiles_pubchem(drug_name):
  """
  Query PubChem for connectivity SMILES by drug name.
  Returns a SMILES string or None if not found.
  """
  try:
    compounds = get_compounds(drug_name, 'name')

    if compounds:
      return compounds[0].connectivity_smiles

  except Exception as e:
    print(f"Error for {drug_name}: {e}")

  return None

In [69]:
if os.path.exists(SMILES_CACHE):
  with open(SMILES_CACHE) as f:
    smiles_cache = json.load(f)
  print(f"loaded SMILES cache: {len(smiles_cache)} entries")
else:
  smiles_cache = {}
  print("no cache found - fetch from PubChem")

loaded SMILES cache: 286 entries


In [73]:
smiles_cache = {}

with open(SMILES_CACHE, 'w') as f:
  json.dump(smiles_cache, f)
  print("SMILES cache cleared.")

SMILES cache cleared.


In [74]:
drug_names = drugs['DRUG_NAME'].dropna().unique().tolist()

n_to_fetch = sum(name not in smiles_cache for name in drug_names)

print(f"drugs to fetch: {n_to_fetch} "
  f"(cached: {len(drug_names) - n_to_fetch})")

failed = []

for i, name in enumerate(drug_names):
  if name in smiles_cache:
    continue

  smiles = fetch_smiles_pubchem(name)
  time.sleep(0.3)
  smiles_cache[name] = smiles if smiles else "NOT_FOUND"

  if smiles is None:
    failed.append(name)

  if (i + 1) % 25 == 0:
    print(f"fetched {i+1}/{len(drug_names)}")

with open(SMILES_CACHE, "w") as f:
  json.dump(smiles_cache, f, indent=2)

print("\nFetch complete")
print(f"Total drugs:   {len(drug_names)}")
print(f"Found SMILES: {sum(v not in [None, 'NOT_FOUND'] for v in smiles_cache.values())}")
print(f"Failed: {sum(v == 'NOT_FOUND' for v in smiles_cache.values())}")

drugs to fetch: 286 (cached: 0)
fetched 25/286
fetched 50/286
fetched 75/286
fetched 100/286
fetched 125/286
fetched 150/286
fetched 175/286
fetched 200/286
fetched 225/286
fetched 250/286
fetched 275/286

Fetch complete
Total drugs:   286
Found SMILES: 229
Failed: 57


In [75]:
failed_drugs = [n for n in drug_names if smiles_cache.get(n) in [None, "NOT_FOUND"]]
print(f'drugs with no SMILES found ({len(failed_drugs)}):')
for name in failed_drugs:
  # find the DRUG_ID
  row = drugs[drugs['DRUG_NAME'] == name]
  print(f'  DRUG_ID={row["DRUG_ID"].values[0] if len(row) else "?"}: {name}')

drugs with no SMILES found (57):
  DRUG_ID=1047: Nutlin-3a (-)
  DRUG_ID=1378: Bleomycin (50 uM)
  DRUG_ID=1428: IAP_5620
  DRUG_ID=1584: BDOCA000347a
  DRUG_ID=1585: BDF00022089a
  DRUG_ID=1586: BDILV000379a
  DRUG_ID=1635: Picolinici-acid
  DRUG_ID=1708: CDK9_5576
  DRUG_ID=1709: CDK9_5038
  DRUG_ID=1712: Eg5_9814
  DRUG_ID=1713: ERK_2440
  DRUG_ID=1714: ERK_6604
  DRUG_ID=1716: IRAK4_4710
  DRUG_ID=1718: JAK1_8709
  DRUG_ID=1730: PAK_5339
  DRUG_ID=1732: TAF1_5496
  DRUG_ID=1733: ULK1_4989
  DRUG_ID=1734: VSP34_8731
  DRUG_ID=1738: IGF1R_3801
  DRUG_ID=1739: JAK_8517
  DRUG_ID=1776: GSK2256098C
  DRUG_ID=1777: GSK2276186C
  DRUG_ID=1779: GSK626616AC
  DRUG_ID=1780: GSK3337463A
  DRUG_ID=1782: GSK2830371A
  DRUG_ID=1783: LMB_AB1
  DRUG_ID=1784: LMB_AB2
  DRUG_ID=1785: LMB_AB3
  DRUG_ID=1820: 123829
  DRUG_ID=1821: 765771
  DRUG_ID=1824: 123138
  DRUG_ID=1826: 50869
  DRUG_ID=1828: 720427
  DRUG_ID=1829: 667880
  DRUG_ID=1832: 729189
  DRUG_ID=1833: 741909
  DRUG_ID=1834: 743380
  DRU

The following drugs failed because the text name in the GDSC database contains typos, salt names, or stereochemical tags (like (-) or (50 uM)) that break exact-match.

In [83]:
# manual corrections for drugs PubChem under a different name
name_corrections = {
    'Nutlin-3a (-)':          'Nutlin-3a',
    'Bleomycin (50 uM)':      'Bleomycin',
    'Picolinici-acid':        'Picolinic acid',
    'ascorbate (vitamin C)':  'Ascorbic acid',
    'GSK-LSD1-2HCl':         'GSK-LSD1',   # the "2HCl" suffix stands for dihydrochloride (the salt formulation used for stability in assays)
    'KRAS (G12C) Inhibitor-12': 'MRTX849',
    'GSK626616AC' :'GSK626616',
    'GSK2256098C' : 'GSK2256098',
    'GSK2276186C' : 'GSK2276186',
    'GSK3337463A' : 'GSK3337463',
    'GSK2830371A' : 'GSK2830371',
    #'VTP-A' : 'VTP',
}

# retry with corrected names
for original, corrected in name_corrections.items():
  smiles = fetch_smiles_pubchem(corrected)
  if smiles:
    smiles_cache[original] = smiles
    print(f'  fixed: {original} - found via "{corrected}"')
  else:
    print(f'  failed: {original} (tried "{corrected}")')

  fixed: Nutlin-3a (-) - found via "Nutlin-3a"
  fixed: Bleomycin (50 uM) - found via "Bleomycin"
  fixed: Picolinici-acid - found via "Picolinic acid"
  fixed: ascorbate (vitamin C) - found via "Ascorbic acid"
  fixed: GSK-LSD1-2HCl - found via "GSK-LSD1"
  fixed: KRAS (G12C) Inhibitor-12 - found via "MRTX849"
  fixed: GSK626616AC - found via "GSK626616"
  fixed: GSK2256098C - found via "GSK2256098"
  failed: GSK2276186C (tried "GSK2276186")
  failed: GSK3337463A (tried "GSK3337463")
  fixed: GSK2830371A - found via "GSK2830371"


In [84]:
failed_drugs = [n for n in drug_names if smiles_cache.get(n) in [None, "NOT_FOUND"]]
print(f'drugs with no SMILES found ({len(failed_drugs)}):')
for name in failed_drugs:
  # find the DRUG_ID
  row = drugs[drugs['DRUG_NAME'] == name]
  print(f'  DRUG_ID={row["DRUG_ID"].values[0] if len(row) else "?"}: {name}')

drugs with no SMILES found (47):
  DRUG_ID=1428: IAP_5620
  DRUG_ID=1584: BDOCA000347a
  DRUG_ID=1585: BDF00022089a
  DRUG_ID=1586: BDILV000379a
  DRUG_ID=1708: CDK9_5576
  DRUG_ID=1709: CDK9_5038
  DRUG_ID=1712: Eg5_9814
  DRUG_ID=1713: ERK_2440
  DRUG_ID=1714: ERK_6604
  DRUG_ID=1716: IRAK4_4710
  DRUG_ID=1718: JAK1_8709
  DRUG_ID=1730: PAK_5339
  DRUG_ID=1732: TAF1_5496
  DRUG_ID=1733: ULK1_4989
  DRUG_ID=1734: VSP34_8731
  DRUG_ID=1738: IGF1R_3801
  DRUG_ID=1739: JAK_8517
  DRUG_ID=1777: GSK2276186C
  DRUG_ID=1780: GSK3337463A
  DRUG_ID=1783: LMB_AB1
  DRUG_ID=1784: LMB_AB2
  DRUG_ID=1785: LMB_AB3
  DRUG_ID=1820: 123829
  DRUG_ID=1821: 765771
  DRUG_ID=1824: 123138
  DRUG_ID=1826: 50869
  DRUG_ID=1828: 720427
  DRUG_ID=1829: 667880
  DRUG_ID=1832: 729189
  DRUG_ID=1833: 741909
  DRUG_ID=1834: 743380
  DRUG_ID=1836: 150412
  DRUG_ID=1839: 615590
  DRUG_ID=1840: 630600
  DRUG_ID=1844: 776928
  DRUG_ID=1866: BDP-00009066
  DRUG_ID=1998: BPD-00008900
  DRUG_ID=1999: N25720-51-A1
  DRUG

### Assess coverage

In [88]:
drugs['smiles'] = drugs['DRUG_NAME'].map(smiles_cache)
drugs['has_smiles'] = drugs['smiles'].apply(
  lambda x: x is not None and x != 'NOT_FOUND' and pd.notna(x)
)
print('SMILES coverage:')
print(f'  drugs with SMILES:    {drugs["has_smiles"].sum()} / {len(drugs)}')
print(f'  drugs without SMILES: {(~drugs["has_smiles"]).sum()} / {len(drugs)}')

# count training pairs affected
pairs_with_smiles = master_pairs.merge(
  drugs[['DRUG_NAME', 'has_smiles']], on='DRUG_NAME', how='left'
)

pairs_with_smiles['has_smiles'] = pairs_with_smiles['has_smiles'].fillna(False).astype(bool)
n_total = len(pairs_with_smiles)
n_covered = pairs_with_smiles['has_smiles'].sum()
n_lost = n_total - n_covered

print(f'\ntraining pairs:')
print(f'  total:               {n_total:,}')
print(f'  covered by SMILES:   {n_covered:,}  ({100*n_covered/n_total:.1f}%)')
print(f'  lost (no SMILES):    {n_lost:,}  ({100*n_lost/n_total:.1f}%)')
print()
print('pairs lost per missing drug:')
lost_by_drug = pairs_with_smiles[~pairs_with_smiles['has_smiles']].groupby('DRUG_NAME').size()
print(lost_by_drug.sort_values(ascending=False).to_string())

SMILES coverage:
  drugs with SMILES:    248 / 295
  drugs without SMILES: 47 / 295

training pairs:
  total:               191,590
  covered by SMILES:   165,433  (86.3%)
  lost (no SMILES):    26,157  (13.7%)

pairs lost per missing drug:
DRUG_NAME
N30652-18-1     576
N29087-69-1     576
N27922-53-1     576
N25720-51-A1    576
BDP-00009066    576
BPD-00008900    576
CT7033-2        574
PBD-288         574
VTP-B           571
THR-101         567
THR-102         567
THR-103         567
ERK_6604        553
CDK9_5576       553
ERK_2440        553
JAK1_8709       553
VSP34_8731      553
JAK_8517        553
IRAK4_4710      553
CDK9_5038       553
ULK1_4989       552
TAF1_5496       552
IGF1R_3801      552
PAK_5339        552
IAP_5620        551
615590          550
630600          550
150412          550
123829          550
123138          550
50869           550
667880          550
BDILV000379a    550
BDF00022089a    550
765771          550
743380          550
741909          550
729189   

### Handle Missing SMILES

In [104]:
# dropping
pct_lost = 100 * n_lost / n_total
print(pct_lost)
master_pairs_smiles = master_pairs[
  master_pairs['DRUG_NAME'].isin(drugs[drugs['has_smiles']]['DRUG_NAME'])
  ].copy()
master_pairs_smiles['smiles'] = master_pairs_smiles['DRUG_NAME'].map(smiles_cache)

print(f'\nmaster_pairs after SMILES filter: {master_pairs_smiles.shape}')
print(f'  cell lines: {master_pairs_smiles["ModelID"].nunique()}')
print(f'  drugs:      {master_pairs_smiles["DRUG_NAME"].nunique()}')

13.652591471371156

master_pairs after SMILES filter: (154357, 10)
  cell lines: 713
  drugs:      239
